## ziwen combine with kevin part
### registration phase (user input student id & email + face detection)
i updated the flow and UI to make it more pretty and easy to use 

In [ ]:
import cv2
import pytesseract
import face_recognition
import re
import os
import csv
import numpy as np
from PIL import Image, ImageTk
import tkinter as tk
from tkinter import messagebox, filedialog
from tkinter import ttk
import math
import time

# Set Tesseract executable path for OCR
pytesseract.pytesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Global image holder
current_image = None

# Cleanup student ID string
def clean_student_id(ocr_id):
    return (
        ocr_id.upper()
        .replace("O9", "09")
        .replace("O", "0", 1)
        .replace("I", "1")
        .replace("S", "5")
    )

# Extract name and ID using OCR from image
def extract_name_and_id(image_pil):
    img_cv = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    sharpened = cv2.addWeighted(gray, 1.5, blur, -0.5, 0)
    thresh = cv2.adaptiveThreshold(sharpened, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 15, 10)
    text = pytesseract.image_to_string(thresh)
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    name = ''
    student_id = ''
    for i, line in enumerate(lines):
        line_cleaned = line.replace("O", "0").replace("S", "5")
        match = re.search(r'\d{2}[A-Z0-9]{3}\d{5}', line_cleaned)
        if match:
            raw_id = match.group()
            student_id = raw_id.replace("0", "O", 1)
            for j in range(max(0, i - 3), i):
                candidate = lines[j]
                if (not re.search(r'\d', candidate) and len(candidate.split()) >= 2 and candidate.isupper()
                        and not any(k in candidate for k in ['DATE', 'EXPIRY', 'STUDENT', 'TARUMT'])):
                    name = candidate
                    break
            break
    return name.strip(), student_id.strip()

# Main logic to confirm OCR results and start live face capture

def confirm_image():
    global current_image
    if current_image is None:
        messagebox.showwarning("No Image", "Please select or capture an image first.")
        return

    # Extract from OCR
    name, student_id = extract_name_and_id(current_image)
    if not name and not student_id:
        messagebox.showwarning("OCR Failed", "Could not extract name or student ID.")
        return

    # Pop-up window for editing name, ID, and email
    edit_window = tk.Toplevel(root)
    edit_window.title("Confirm Student Info")
    ttk.Label(edit_window, text="Please confirm or edit your information below:").grid(row=0, column=0, columnspan=2, pady=(10, 5))

    def force_uppercase_name(*args):
        current = name_var.get()
        name_var.set(current.upper())

    name_var = tk.StringVar()
    name_var.trace_add("write", force_uppercase_name)

    # Editable entries for student info
    ttk.Label(edit_window, text="Name:").grid(row=1, column=0, padx=10, pady=5, sticky="e")
    name_entry = ttk.Entry(edit_window, width=40, textvariable=name_var)
    name_var.set(name)
    name_entry.grid(row=1, column=1, padx=10, pady=5)

    ttk.Label(edit_window, text="Student ID:").grid(row=2, column=0, padx=10, pady=5, sticky="e")
    id_entry = ttk.Entry(edit_window, width=40)
    corrected_id = clean_student_id(student_id)
    id_entry.insert(0, corrected_id)
    id_entry.grid(row=2, column=1, padx=10, pady=5)

    ttk.Label(edit_window, text="School Email:").grid(row=3, column=0, padx=10, pady=5, sticky="e")
    email_entry = ttk.Entry(edit_window, width=40)
    email_entry.grid(row=3, column=1, padx=10, pady=5)

    # Save student data and launch live face capture process
    def save_data():
        final_name = name_entry.get().strip().replace(" ", "_")
        final_id = id_entry.get().strip()
        email = email_entry.get().strip()
        if not final_name or not final_id or not email:
            messagebox.showerror("Error", "Name, Student ID, and Email are required.")
            return

        folder_name = os.path.join("StudentidFolder", f"{final_name}.{final_id}")
        temp_folder_created = False
        buffer_encodings = []
        frame_count = 0
        capture_interval = 10
        timeout_seconds = 60
        start_time = time.time()

        cap = cv2.VideoCapture(0)
        from mtcnn import MTCNN
        detector = MTCNN()

        while True:
            ret, frame = cap.read()
            if not ret:
                continue

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = detector.detect_faces(rgb)
            h, w, _ = frame.shape
            center = (w // 2, h // 2)
            radius = 140
            segments = 40
            completed_ratio = len(buffer_encodings) / 7

            # Draw progress ring segments
            for i in range(segments):
                angle = 2 * math.pi * i / segments
                x1 = int(center[0] + radius * math.cos(angle))
                y1 = int(center[1] + radius * math.sin(angle))
                x2 = int(center[0] + (radius + 12) * math.cos(angle))
                y2 = int(center[1] + (radius + 12) * math.sin(angle))
                if i < math.floor(completed_ratio * segments):
                    cv2.line(frame, (x1, y1), (x2, y2), (0, 200, 0), 2)
                else:
                    cv2.line(frame, (x1, y1), (x2, y2), (180, 180, 180), 2)

            cv2.circle(frame, center, radius - 10, (255, 255, 255), 2)
            cv2.putText(frame, "Move your head slowly to complete the circle", (center[0] - 200, center[1] + radius + 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            # Face detection & encoding collection
            if len(results) == 1:
                if not temp_folder_created:
                    os.makedirs(folder_name, exist_ok=True)
                    temp_folder_created = True
                    img_path = os.path.join(folder_name, f"{final_name}.{final_id}.jpg")
                    current_image.save(img_path)
                if frame_count % capture_interval == 0:
                    x, y, w_box, h_box = results[0]['box']
                    top, right, bottom, left = y, x + w_box, y + h_box, x
                    encodings = face_recognition.face_encodings(rgb, known_face_locations=[(top, right, bottom, left)])
                    if encodings:
                        buffer_encodings.append(encodings[0])
                        if len(buffer_encodings) >= 7:
                            mean_encoding = np.mean(buffer_encodings, axis=0)
                            np.save(os.path.join(folder_name, "face_encoding.npy"), mean_encoding)
                            with open("student_records.csv", "a", newline="") as f:
                                writer = csv.writer(f)
                                if os.stat("student_records.csv").st_size == 0:
                                    writer.writerow(["Name", "Student ID", "Email", "Image Path"])
                                writer.writerow([final_name.replace("_", " "), final_id, email, img_path])
                            messagebox.showinfo("Success", "RRegistration complete! You may now proceed to register the next student.")
                            break
            elif len(results) > 1:
                cv2.putText(frame, "Multiple faces detected", (60, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            frame_count += 1
            if time.time() - start_time > timeout_seconds:
                if temp_folder_created:
                    try:
                        for f in os.listdir(folder_name):
                            os.remove(os.path.join(folder_name, f))
                        os.rmdir(folder_name)
                    except:
                        pass
                messagebox.showwarning("Timeout", "No face detected in time. Registration failed. Please try again.")
                break

            cv2.imshow("Live Face Capture", frame)
            if cv2.waitKey(1) & 0xFF == 27:
                break

        cap.release()
        cv2.destroyAllWindows()
        edit_window.destroy()
        retake_or_reselect()

    ttk.Button(edit_window, text="Confirm & Save", command=save_data).grid(row=4, column=0, columnspan=2, pady=10)

# Reset GUI after each registration
def retake_or_reselect():
    global current_image
    panel.config(image=None)
    panel.image = None
    current_image = None
    confirm_btn.pack_forget()
    retake_btn.pack_forget()
    upload_btn.pack(pady=10)
    capture_btn.pack(pady=5)

# GUI setup
root = tk.Tk()
root.title("🎓 Convocation Registration System")
root.geometry("480x600")
welcome = ttk.Label(root, text="Welcome to Convocation Registration System!", font=("Helvetica", 14, "bold"))
welcome.pack(pady=10)
instruction = ttk.Label(root, text="Please register using your student ID card (upload or webcam)")
instruction.pack()
upload_btn = ttk.Button(root, text="Upload Student ID Card", command=upload_image)
upload_btn.pack(pady=10)
capture_btn = ttk.Button(root, text="Capture Student ID via Webcam", command=take_picture)
capture_btn.pack(pady=5)
panel = ttk.Label(root)
panel.pack(padx=10, pady=10)
confirm_btn = ttk.Button(root, text="Confirm ID Card", command=confirm_image)
retake_btn = ttk.Button(root, text="Retake / Select Another Image", command=retake_or_reselect)
root.mainloop()


### ceremony day (face verfication)

In [ ]:
import cv2
import numpy as np
import face_recognition
import os

# Step 1: Load all registered student encodings
student_encodings = []
student_labels = []
folder_base = "StudentidFolder"

for folder in os.listdir(folder_base):
    folder_path = os.path.join(folder_base, folder)
    encoding_path = os.path.join(folder_path, "face_encoding.npy")
    if os.path.exists(encoding_path):
        encoding = np.load(encoding_path)
        student_encodings.append(encoding)
        student_labels.append(folder)  

if not student_encodings:
    print("=== No registered student encodings found. ===")
    exit()

# Step 2: Start webcam
cap = cv2.VideoCapture(0)
print("📷 Webcam running. Stand in front of the camera to be recognized.")

font = cv2.FONT_HERSHEY_SIMPLEX
recognized_label = ""

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    face_locations = face_recognition.face_locations(rgb)

    for (top, right, bottom, left) in face_locations:
        live_encoding = face_recognition.face_encodings(rgb, [ (top, right, bottom, left) ])[0]
        distances = face_recognition.face_distance(student_encodings, live_encoding)
        best_match_index = np.argmin(distances)
        best_distance = distances[best_match_index]

        if best_distance < 0.40:
            matched_raw = student_labels[best_match_index]
            if "." in matched_raw:
                name, sid = matched_raw.split(".", 1)
                display_text = f"{name.replace('_', ' ')} - {sid}"
            else:
                display_text = matched_raw.replace("_", " ")
            cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
            cv2.putText(frame, f"{display_text}", (left, top - 10), font, 0.8, (0, 255, 0), 2)
        else:
            cv2.rectangle(frame, (left, top), (right, bottom), (0, 0, 255), 2)
            cv2.putText(frame, "Unknown Face", (left, top - 10), font, 0.8, (0, 0, 255), 2)

    cv2.putText(frame, "Press ESC to quit", (10, 30), font, 0.6, (255, 255, 255), 2)
    cv2.imshow("Graduation Face Recognition", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


📷 Webcam running. Stand in front of the camera to be recognized.
